# 04 - Static Pruning 25% Validation/Test

Bu notebook, daha önce seçilen **B4 PatchTST baseline** modeli üzerinde statik %25 head pruning deneyi yapar.

Amaç:

1. B4 checkpoint'ini yüklemek  
2. `static_prune_25_percent_heads.csv` dosyasındaki 6 head'i maskelemek  
3. Validation setinde baseline ve pruned loss karşılaştırmak  
4. Test setinde baseline ve pruned MSE/MAE karşılaştırmak  
5. Sonuçları Drive'a kaydetmek  

Kullanılan pruning maskesi:

- Toplam head: `3 layer × 8 head = 24`
- %25 pruning: `6 head kapalı`
- Head'ler, önceki head importance analizindeki en düşük `overall_importance` değerlerine göre seçilmiştir.


## 1. Drive bağla ve temel importlar

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path
import sys
import os
import shutil
import random
import json

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from tqdm.auto import tqdm

## 2. Proje yolları

In [3]:
PROJECT_DIR = Path(
    "/content/drive/MyDrive/BIL401_Regime_Head_Pruning"
)

REGIME_DIR = PROJECT_DIR / "regime_detection"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
HEAD_IMPORTANCE_DIR = PROJECT_DIR / "head_importance"
PRUNING_DIR = PROJECT_DIR / "pruning_experiments"

PRUNING_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("REGIME_DIR:", REGIME_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("HEAD_IMPORTANCE_DIR:", HEAD_IMPORTANCE_DIR)
print("PRUNING_DIR:", PRUNING_DIR)

PROJECT_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning
REGIME_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/regime_detection
CHECKPOINT_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/checkpoints
HEAD_IMPORTANCE_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/head_importance
PRUNING_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/pruning_experiments


## 3. Time-Series-Library hazırla

Yeni Colab runtime açıldıysa repo yeniden clone edilir. Veri seti Drive'da varsa oradan kopyalanır, yoksa GitHub'dan indirilir.


In [4]:
TSLIB_DIR = Path("/content/Time-Series-Library")

if not TSLIB_DIR.exists():
    %cd /content
    !git clone https://github.com/thuml/Time-Series-Library.git
else:
    print("Time-Series-Library already exists:", TSLIB_DIR)

sys.path.insert(0, str(TSLIB_DIR))
print("Python path[0]:", sys.path[0])

/content
Cloning into 'Time-Series-Library'...
remote: Enumerating objects: 2295, done.
remote: Total 2295 (delta 0), reused 0 (delta 0), pack-reused 2295 (from 1)
Receiving objects: 100% (2295/2295), 78.43 MiB | 16.53 MiB/s, done.
Resolving deltas: 100% (1570/1570), done.
Python path[0]: /content/Time-Series-Library


In [5]:
# TSLib importları için minimal paketler
!pip install -q patool sktime scikit-base --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.5/37.5 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 18.0 MB/s eta 0:00:00


In [6]:
drive_data_path = PROJECT_DIR / "data" / "ETTh1.csv"

tslib_data_path = (
    TSLIB_DIR
    / "dataset/ETDataset/ETT-small/ETTh1.csv"
)

tslib_data_path.parent.mkdir(parents=True, exist_ok=True)

if drive_data_path.exists():
    shutil.copy2(drive_data_path, tslib_data_path)
    print("ETTh1 copied from Drive.")
else:
    print("Drive data not found. Downloading ETTh1...")
    !wget -q https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv -O /content/Time-Series-Library/dataset/ETDataset/ETT-small/ETTh1.csv

print("Dataset exists:", tslib_data_path.exists())
print("Dataset path:", tslib_data_path)

df = pd.read_csv(tslib_data_path)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Drive data not found. Downloading ETTh1...
Dataset exists: True
Dataset path: /content/Time-Series-Library/dataset/ETDataset/ETT-small/ETTh1.csv
Dataset shape: (17420, 8)
Columns: ['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']


,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
0,2016-07-01 00:00:00,5.827,2.009,1.599,0.462,4.203,1.340,30.531000
1,2016-07-01 01:00:00,5.693,2.076,1.492,0.426,4.142,1.371,27.787001
2,2016-07-01 02:00:00,5.157,1.741,1.279,0.355,3.777,1.218,27.787001
3,2016-07-01 03:00:00,5.090,1.942,1.279,0.391,3.807,1.279,25.044001
4,2016-07-01 04:00:00,5.358,1.942,1.492,0.462,3.868,1.279,21.948000


## 4. Regime dosyasını ve pruning listesini oku

In [7]:
regime_path = REGIME_DIR / "etth1_validation_regimes_ot_seq336.csv"
regime_df = pd.read_csv(regime_path)

print("Regime df shape:", regime_df.shape)
display(regime_df.head())
display(regime_df["regime"].value_counts())

Regime df shape: (2785, 13)


,window_id,start_idx,end_idx,input_start_date,input_end_date,trend_score,seasonal_score,residual_score,regime,top_score,second_score,confidence_margin,is_confident
0,0,0,336,2017-06-12 00:00:00,2017-06-25 23:00:00,0.700476,0.205640,0.093884,trend,0.700476,0.205640,0.494836,True
1,1,1,337,2017-06-12 01:00:00,2017-06-26 00:00:00,0.700166,0.206039,0.093796,trend,0.700166,0.206039,0.494127,True
2,2,2,338,2017-06-12 02:00:00,2017-06-26 01:00:00,0.696768,0.208689,0.094543,trend,0.696768,0.208689,0.488079,True
3,3,3,339,2017-06-12 03:00:00,2017-06-26 02:00:00,0.689886,0.213555,0.096559,trend,0.689886,0.213555,0.476330,True
4,4,4,340,2017-06-12 04:00:00,2017-06-26 03:00:00,0.677356,0.223502,0.099142,trend,0.677356,0.223502,0.453854,True


,count
regime,
trend,2134
residual,359
seasonal,292


In [8]:
static_prune_path = (
    HEAD_IMPORTANCE_DIR
    / "summaries"
    / "static_prune_25_percent_heads.csv"
)

static_prune_df = pd.read_csv(static_prune_path)

print("Static prune heads:")
display(static_prune_df[["layer", "head", "overall_importance"]])

assert len(static_prune_df) == 6, "25% pruning için 6 head bekleniyor."


Static prune heads:


,layer,head,overall_importance
0,0,5,-0.011536
1,1,7,-0.010845
2,2,0,-0.009892
3,1,4,-0.008351
4,1,1,-0.005822
5,0,3,-0.005073


## 5. B4 checkpoint yolunu bul

In [9]:
b4_checkpoint_candidates = list(
    CHECKPOINT_DIR.glob(
        "B4_patchtst_etth1_336_dm128_h8/**/checkpoint.pth"
    )
)

print("Found checkpoint candidates:")
for path in b4_checkpoint_candidates:
    print(path)

if len(b4_checkpoint_candidates) == 0:
    raise FileNotFoundError(
        "B4 checkpoint bulunamadı. CHECKPOINT_DIR içini kontrol et."
    )

b4_checkpoint_path = b4_checkpoint_candidates[0]
print("\nSelected checkpoint:")
print(b4_checkpoint_path)

Found checkpoint candidates:
/content/drive/MyDrive/BIL401_Regime_Head_Pruning/checkpoints/B4_patchtst_etth1_336_dm128_h8/checkpoint.pth

Selected checkpoint:
/content/drive/MyDrive/BIL401_Regime_Head_Pruning/checkpoints/B4_patchtst_etth1_336_dm128_h8/checkpoint.pth


## 6. B4 model argümanlarını oluştur

In [14]:
from argparse import Namespace

args = Namespace(
    # task
    task_name="long_term_forecast",
    is_training=0,
    model_id="ETTh1_336_96_dm128_h8",
    model="PatchTST",

    # data
    data="ETTh1",
    root_path="./dataset/ETDataset/ETT-small/",
    data_path="ETTh1.csv",
    features="M",
    target="OT",
    freq="h",
    checkpoints="./checkpoints/",

    # forecasting
    seq_len=336,
    label_len=48,
    pred_len=96,
    seasonal_patterns="Monthly",
    inverse=False,

    # model
    enc_in=7,
    dec_in=7,
    c_out=7,
    d_model=128,
    n_heads=8,
    e_layers=3,
    d_layers=1,
    d_ff=256,
    moving_avg=25,
    factor=3,
    distil=True,
    dropout=0.1,
    embed="timeF",
    activation="gelu",
    output_attention=False,

    # PatchTST related
    patch_len=16,
    stride=8,
    padding_patch="end",
    revin=1,
    affine=0,
    subtract_last=0,
    decomposition=0,
    kernel_size=25,
    individual=0,

    # optimization / loader
    num_workers=0,
    itr=1,
    train_epochs=10,
    batch_size=32,
    patience=3,
    learning_rate=0.0001,
    des="baseline_b4",
    loss="MSE",
    lradj="type1",
    use_amp=False,

    # GPU
    use_gpu=torch.cuda.is_available(),
    gpu=0,
    use_multi_gpu=False,
    devices="0",
    gpu_type="cuda",
    # other model families, required by some imports
    expand=2,
    d_conv=4,
    top_k=5,
    num_kernels=6,
    channel_independence=0,
    decomp_method="moving_avg",
    use_norm=1,
    down_sampling_layers=0,
    down_sampling_window=1,
    down_sampling_method=None,
    seg_len=48,

    # MLP projection args sometimes expected
    p_hidden_dims=[128, 128],
    p_hidden_layers=2,
)

print(args)

Namespace(task_name='long_term_forecast', is_training=0, model_id='ETTh1_336_96_dm128_h8', model='PatchTST', data='ETTh1', root_path='./dataset/ETDataset/ETT-small/', data_path='ETTh1.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=336, label_len=48, pred_len=96, seasonal_patterns='Monthly', inverse=False, enc_in=7, dec_in=7, c_out=7, d_model=128, n_heads=8, e_layers=3, d_layers=1, d_ff=256, moving_avg=25, factor=3, distil=True, dropout=0.1, embed='timeF', activation='gelu', output_attention=False, patch_len=16, stride=8, padding_patch='end', revin=1, affine=0, subtract_last=0, decomposition=0, kernel_size=25, individual=0, num_workers=0, itr=1, train_epochs=10, batch_size=32, patience=3, learning_rate=0.0001, des='baseline_b4', loss='MSE', lradj='type1', use_amp=False, use_gpu=True, gpu=0, use_multi_gpu=False, devices='0', gpu_type='cuda', expand=2, d_conv=4, top_k=5, num_kernels=6, channel_independence=0, decomp_method='moving_avg', use_norm=1, down_s

## 7. Modeli yükle

In [11]:
%cd /content/Time-Series-Library

/content/Time-Series-Library


In [16]:
!pip install reformer-pytorch --no-deps
!pip install local-attention --no-deps
!pip install hyper_connections --no-deps
!pip install axial_positional_embedding --no-deps
!pip install product_key_memory --no-deps
!pip install colt5_attention --no-deps

In [17]:
from exp.exp_long_term_forecasting import Exp_Long_Term_Forecast

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", device)

exp = Exp_Long_Term_Forecast(args)
model = exp.model.to(device)

checkpoint = torch.load(
    b4_checkpoint_path,
    map_location=device,
)

model.load_state_dict(checkpoint)
model.eval()

print("Model loaded successfully.")

Device: cuda:0
Use GPU: cuda:0
🚀 Lazy Loading: PatchTST ...
Model loaded successfully.


## 8. Validation ve test loader oluştur

In [18]:
from data_provider.data_factory import data_provider

vali_data, vali_loader = data_provider(
    args,
    flag="val",
)

test_data, test_loader = data_provider(
    args,
    flag="test",
)

print("Validation dataset length:", len(vali_data))
print("Validation loader batches:", len(vali_loader))
print("Test dataset length:", len(test_data))
print("Test loader batches:", len(test_loader))

assert len(vali_data) == len(regime_df), (
    len(vali_data),
    len(regime_df),
)

print("Validation windows and regime labels match.")

val 2785
test 2785
Validation dataset length: 2785
Validation loader batches: 88
Test dataset length: 2785
Test loader batches: 88
Validation windows and regime labels match.


## 9. HeadMaskController

Bu sınıf, `AttentionLayer.forward` fonksiyonunu maske destekli hale getirir.

Maske boyutu:

```text
[num_layers, num_heads] = [3, 8]
```

`1.0` aktif head, `0.0` kapalı head anlamına gelir.


In [19]:
class HeadMaskController:
    def __init__(self, model):
        self.model = model
        self.original_forwards = {}
        self.current_mask = None

    def install(self):
        for layer_idx, encoder_layer in enumerate(self.model.encoder.attn_layers):
            attention_layer = encoder_layer.attention

            if layer_idx in self.original_forwards:
                continue

            original_forward = attention_layer.forward
            self.original_forwards[layer_idx] = original_forward

            def make_masked_forward(layer_idx, attention_layer):
                def masked_forward(
                    queries,
                    keys,
                    values,
                    attn_mask,
                    tau=None,
                    delta=None,
                ):
                    B, L, _ = queries.shape
                    _, S, _ = keys.shape
                    H = attention_layer.n_heads

                    queries_proj = attention_layer.query_projection(queries)
                    keys_proj = attention_layer.key_projection(keys)
                    values_proj = attention_layer.value_projection(values)

                    queries_proj = queries_proj.view(B, L, H, -1)
                    keys_proj = keys_proj.view(B, S, H, -1)
                    values_proj = values_proj.view(B, S, H, -1)

                    out, attn = attention_layer.inner_attention(
                        queries_proj,
                        keys_proj,
                        values_proj,
                        attn_mask,
                        tau=tau,
                        delta=delta,
                    )

                    if self.current_mask is not None:
                        layer_mask = self.current_mask[layer_idx].to(out.device)
                        layer_mask = layer_mask.view(1, 1, H, 1)
                        out = out * layer_mask

                    out = out.view(B, L, -1)

                    return attention_layer.out_projection(out), attn

                return masked_forward

            attention_layer.forward = make_masked_forward(
                layer_idx,
                attention_layer,
            )

    def remove(self):
        for layer_idx, original_forward in self.original_forwards.items():
            self.model.encoder.attn_layers[layer_idx].attention.forward = original_forward

        self.original_forwards = {}
        self.current_mask = None

    def set_all_active(self):
        num_layers = len(self.model.encoder.attn_layers)
        num_heads = self.model.encoder.attn_layers[0].attention.n_heads

        self.current_mask = torch.ones(
            num_layers,
            num_heads,
            dtype=torch.float32,
        )

    def set_mask(self, mask):
        self.current_mask = mask.clone().float()

    def build_static_prune_mask(self, prune_df):
        self.set_all_active()

        for _, row in prune_df.iterrows():
            layer_idx = int(row["layer"])
            head_idx = int(row["head"])
            self.current_mask[layer_idx, head_idx] = 0.0

        return self.current_mask.clone()

In [20]:
mask_controller = HeadMaskController(model)
mask_controller.install()

num_layers = len(model.encoder.attn_layers)
num_heads = model.encoder.attn_layers[0].attention.n_heads

print("Layers:", num_layers)
print("Heads per layer:", num_heads)
print("Total heads:", num_layers * num_heads)

baseline_mask = torch.ones(num_layers, num_heads)
static_prune_mask = mask_controller.build_static_prune_mask(static_prune_df)

print("Static prune mask:")
print(static_prune_mask)

print("Pruned head count:", int((static_prune_mask == 0).sum().item()))

Layers: 3
Heads per layer: 8
Total heads: 24
Static prune mask:
tensor([[1., 1., 1., 0., 1., 0., 1., 1.],
        [1., 0., 1., 1., 0., 1., 1., 0.],
        [0., 1., 1., 1., 1., 1., 1., 1.]])
Pruned head count: 6


## 10. Mask test

Tek bir batch üzerinde baseline ve static-pruned output farkını kontrol edelim. Fark sıfırdan büyük olmalı.


In [21]:
batch = next(iter(vali_loader))
batch_x, batch_y, batch_x_mark, batch_y_mark = batch

batch_x = batch_x.float().to(device)
batch_y = batch_y.float().to(device)
batch_x_mark = batch_x_mark.float().to(device)
batch_y_mark = batch_y_mark.float().to(device)

model.eval()

mask_controller.set_mask(baseline_mask)

with torch.no_grad():
    outputs_baseline = model(
        batch_x,
        batch_x_mark,
        batch_y,
        batch_y_mark,
    )

mask_controller.set_mask(static_prune_mask)

with torch.no_grad():
    outputs_static_pruned = model(
        batch_x,
        batch_x_mark,
        batch_y,
        batch_y_mark,
    )

diff = torch.mean(
    torch.abs(outputs_baseline - outputs_static_pruned)
).item()

print("Baseline output shape:", outputs_baseline.shape)
print("Static-pruned output shape:", outputs_static_pruned.shape)
print("Mean absolute output difference:", diff)

assert diff > 0, "Mask output'u değiştirmedi. Head masking çalışmıyor olabilir."

mask_controller.set_mask(baseline_mask)

Baseline output shape: torch.Size([32, 96, 7])
Static-pruned output shape: torch.Size([32, 96, 7])
Mean absolute output difference: 0.11248654872179031


## 11. Loss hesaplama fonksiyonları

In [22]:
mse_criterion = nn.MSELoss(reduction="none")
mae_criterion = nn.L1Loss(reduction="none")

def compute_regime_losses_with_mask(
    model,
    loader,
    regime_df,
    mask_controller,
    mask,
    device,
    pred_len=96,
    desc="validation",
):
    model.eval()
    mask_controller.set_mask(mask)

    all_records = []
    global_index = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc=desc):
            batch_x, batch_y, batch_x_mark, batch_y_mark = batch

            batch_x = batch_x.float().to(device)
            batch_y = batch_y.float().to(device)
            batch_x_mark = batch_x_mark.float().to(device)
            batch_y_mark = batch_y_mark.float().to(device)

            outputs = model(
                batch_x,
                batch_x_mark,
                batch_y,
                batch_y_mark,
            )

            true = batch_y[:, -pred_len:, :]

            mse_per_sample = mse_criterion(outputs, true).mean(dim=(1, 2))
            mae_per_sample = mae_criterion(outputs, true).mean(dim=(1, 2))

            batch_size = batch_x.shape[0]

            for i in range(batch_size):
                window_id = global_index + i
                regime = regime_df.iloc[window_id]["regime"]

                all_records.append({
                    "window_id": window_id,
                    "regime": regime,
                    "mse": float(mse_per_sample[i].detach().cpu()),
                    "mae": float(mae_per_sample[i].detach().cpu()),
                })

            global_index += batch_size

    result_df = pd.DataFrame(all_records)

    regime_summary = (
        result_df
        .groupby("regime")
        .agg(
            mse=("mse", "mean"),
            mae=("mae", "mean"),
            count=("window_id", "count"),
        )
        .reset_index()
    )

    overall = {
        "overall_mse": float(result_df["mse"].mean()),
        "overall_mae": float(result_df["mae"].mean()),
    }

    return {
        "overall": overall,
        "regime_summary": regime_summary,
        "window_losses": result_df,
    }


def compute_test_metrics_with_mask(
    model,
    loader,
    mask_controller,
    mask,
    device,
    pred_len=96,
    desc="test",
):
    model.eval()
    mask_controller.set_mask(mask)

    all_mse = []
    all_mae = []

    with torch.no_grad():
        for batch in tqdm(loader, desc=desc):
            batch_x, batch_y, batch_x_mark, batch_y_mark = batch

            batch_x = batch_x.float().to(device)
            batch_y = batch_y.float().to(device)
            batch_x_mark = batch_x_mark.float().to(device)
            batch_y_mark = batch_y_mark.float().to(device)

            outputs = model(
                batch_x,
                batch_x_mark,
                batch_y,
                batch_y_mark,
            )

            true = batch_y[:, -pred_len:, :]

            mse_per_sample = mse_criterion(outputs, true).mean(dim=(1, 2))
            mae_per_sample = mae_criterion(outputs, true).mean(dim=(1, 2))

            all_mse.extend(mse_per_sample.detach().cpu().numpy().tolist())
            all_mae.extend(mae_per_sample.detach().cpu().numpy().tolist())

    return {
        "test_mse": float(np.mean(all_mse)),
        "test_mae": float(np.mean(all_mae)),
    }


## 12. Validation karşılaştırması

Önce hiçbir head kapalı değilken B4 validation loss'u, sonra %25 static pruning maskesiyle validation loss hesaplanır.


In [23]:
baseline_val = compute_regime_losses_with_mask(
    model=model,
    loader=vali_loader,
    regime_df=regime_df,
    mask_controller=mask_controller,
    mask=baseline_mask,
    device=device,
    pred_len=args.pred_len,
    desc="Baseline validation",
)

static_pruned_val = compute_regime_losses_with_mask(
    model=model,
    loader=vali_loader,
    regime_df=regime_df,
    mask_controller=mask_controller,
    mask=static_prune_mask,
    device=device,
    pred_len=args.pred_len,
    desc="Static 25% pruned validation",
)

print("Baseline validation overall:")
print(baseline_val["overall"])

print("\nStatic-pruned validation overall:")
print(static_pruned_val["overall"])

print("\nBaseline regime summary:")
display(baseline_val["regime_summary"])

print("\nStatic-pruned regime summary:")
display(static_pruned_val["regime_summary"])

Baseline validation:   0%|          | 0/88 [00:00<?, ?it/s]

Static 25% pruned validation:   0%|          | 0/88 [00:00<?, ?it/s]

Baseline validation overall:
{'overall_mse': 0.6780664091237572, 'overall_mae': 0.5550830315644694}

Static-pruned validation overall:
{'overall_mse': 0.6536577505312017, 'overall_mae': 0.5506856776546533}

Baseline regime summary:


,regime,mse,mae,count
0,residual,0.685031,0.554888,359
1,seasonal,0.660362,0.547936,292
2,trend,0.679317,0.556094,2134



Static-pruned regime summary:


,regime,mse,mae,count
0,residual,0.634596,0.547463,359
1,seasonal,0.689755,0.561695,292
2,trend,0.651925,0.549721,2134


In [24]:
baseline_val_overall = baseline_val["overall"]
static_val_overall = static_pruned_val["overall"]

val_comparison = pd.DataFrame([
    {
        "setting": "B4_no_pruning",
        "pruning_type": "none",
        "pruned_heads": 0,
        "active_heads": 24,
        "pruning_ratio": 0.0,
        "validation_mse": baseline_val_overall["overall_mse"],
        "validation_mae": baseline_val_overall["overall_mae"],
    },
    {
        "setting": "B4_static_pruning_25",
        "pruning_type": "static_overall_importance",
        "pruned_heads": int((static_prune_mask == 0).sum().item()),
        "active_heads": int((static_prune_mask == 1).sum().item()),
        "pruning_ratio": float((static_prune_mask == 0).sum().item() / static_prune_mask.numel()),
        "validation_mse": static_val_overall["overall_mse"],
        "validation_mae": static_val_overall["overall_mae"],
    },
])

baseline_mse = val_comparison.loc[
    val_comparison["setting"] == "B4_no_pruning",
    "validation_mse",
].iloc[0]

baseline_mae = val_comparison.loc[
    val_comparison["setting"] == "B4_no_pruning",
    "validation_mae",
].iloc[0]

val_comparison["delta_validation_mse"] = (
    val_comparison["validation_mse"] - baseline_mse
)

val_comparison["delta_validation_mae"] = (
    val_comparison["validation_mae"] - baseline_mae
)

val_comparison["relative_mse_change_percent"] = (
    val_comparison["delta_validation_mse"]
    / baseline_mse
    * 100
)

val_comparison["relative_mae_change_percent"] = (
    val_comparison["delta_validation_mae"]
    / baseline_mae
    * 100
)

display(val_comparison)

,setting,pruning_type,pruned_heads,active_heads,pruning_ratio,validation_mse,validation_mae,delta_validation_mse,delta_validation_mae,relative_mse_change_percent,relative_mae_change_percent
0,B4_no_pruning,none,0,24,0.00,0.678066,0.555083,0.000000,0.000000,0.000000,0.000000
1,B4_static_pruning_25,static_overall_importance,6,18,0.25,0.653658,0.550686,-0.024409,-0.004397,-3.599745,-0.792198


## 13. Regime bazında validation karşılaştırması

In [25]:
baseline_regime = baseline_val["regime_summary"].copy()
baseline_regime["setting"] = "B4_no_pruning"

static_regime = static_pruned_val["regime_summary"].copy()
static_regime["setting"] = "B4_static_pruning_25"

regime_val_comparison = pd.concat(
    [baseline_regime, static_regime],
    ignore_index=True,
)

display(regime_val_comparison)

# Delta tablo
baseline_regime_ref = baseline_regime[
    ["regime", "mse", "mae"]
].rename(
    columns={
        "mse": "baseline_mse",
        "mae": "baseline_mae",
    }
)

static_regime_delta = static_regime.merge(
    baseline_regime_ref,
    on="regime",
    how="left",
)

static_regime_delta["delta_mse"] = (
    static_regime_delta["mse"]
    - static_regime_delta["baseline_mse"]
)

static_regime_delta["delta_mae"] = (
    static_regime_delta["mae"]
    - static_regime_delta["baseline_mae"]
)

static_regime_delta["relative_mse_change_percent"] = (
    static_regime_delta["delta_mse"]
    / static_regime_delta["baseline_mse"]
    * 100
)

static_regime_delta["relative_mae_change_percent"] = (
    static_regime_delta["delta_mae"]
    / static_regime_delta["baseline_mae"]
    * 100
)

display(static_regime_delta)

,regime,mse,mae,count,setting
0,residual,0.685031,0.554888,359,B4_no_pruning
1,seasonal,0.660362,0.547936,292,B4_no_pruning
2,trend,0.679317,0.556094,2134,B4_no_pruning
3,residual,0.634596,0.547463,359,B4_static_pruning_25
4,seasonal,0.689755,0.561695,292,B4_static_pruning_25
5,trend,0.651925,0.549721,2134,B4_static_pruning_25


,regime,mse,mae,count,setting,baseline_mse,baseline_mae,delta_mse,delta_mae,relative_mse_change_percent,relative_mae_change_percent
0,residual,0.634596,0.547463,359,B4_static_pruning_25,0.685031,0.554888,-0.050435,-0.007425,-7.362383,-1.338145
1,seasonal,0.689755,0.561695,292,B4_static_pruning_25,0.660362,0.547936,0.029393,0.013759,4.451061,2.511107
2,trend,0.651925,0.549721,2134,B4_static_pruning_25,0.679317,0.556094,-0.027392,-0.006372,-4.032308,-1.145920


## 14. Test seti karşılaştırması

Burada regime etiketi kullanmıyoruz. Aynı static pruning maskesi test setinde de uygulanır.


In [26]:
baseline_test = compute_test_metrics_with_mask(
    model=model,
    loader=test_loader,
    mask_controller=mask_controller,
    mask=baseline_mask,
    device=device,
    pred_len=args.pred_len,
    desc="Baseline test",
)

static_pruned_test = compute_test_metrics_with_mask(
    model=model,
    loader=test_loader,
    mask_controller=mask_controller,
    mask=static_prune_mask,
    device=device,
    pred_len=args.pred_len,
    desc="Static 25% pruned test",
)

print("Baseline test:")
print(baseline_test)

print("\nStatic-pruned test:")
print(static_pruned_test)

Baseline test:   0%|          | 0/88 [00:00<?, ?it/s]

Static 25% pruned test:   0%|          | 0/88 [00:00<?, ?it/s]

Baseline test:
{'test_mse': 0.3725455730083387, 'test_mae': 0.39821428153630434}

Static-pruned test:
{'test_mse': 0.37831396463208394, 'test_mae': 0.40442411937020195}


In [27]:
test_comparison = pd.DataFrame([
    {
        "setting": "B4_no_pruning",
        "pruning_type": "none",
        "pruned_heads": 0,
        "active_heads": 24,
        "pruning_ratio": 0.0,
        "test_mse": baseline_test["test_mse"],
        "test_mae": baseline_test["test_mae"],
    },
    {
        "setting": "B4_static_pruning_25",
        "pruning_type": "static_overall_importance",
        "pruned_heads": int((static_prune_mask == 0).sum().item()),
        "active_heads": int((static_prune_mask == 1).sum().item()),
        "pruning_ratio": float((static_prune_mask == 0).sum().item() / static_prune_mask.numel()),
        "test_mse": static_pruned_test["test_mse"],
        "test_mae": static_pruned_test["test_mae"],
    },
])

baseline_test_mse = test_comparison.loc[
    test_comparison["setting"] == "B4_no_pruning",
    "test_mse",
].iloc[0]

baseline_test_mae = test_comparison.loc[
    test_comparison["setting"] == "B4_no_pruning",
    "test_mae",
].iloc[0]

test_comparison["delta_test_mse"] = (
    test_comparison["test_mse"] - baseline_test_mse
)

test_comparison["delta_test_mae"] = (
    test_comparison["test_mae"] - baseline_test_mae
)

test_comparison["relative_mse_change_percent"] = (
    test_comparison["delta_test_mse"]
    / baseline_test_mse
    * 100
)

test_comparison["relative_mae_change_percent"] = (
    test_comparison["delta_test_mae"]
    / baseline_test_mae
    * 100
)

display(test_comparison)

,setting,pruning_type,pruned_heads,active_heads,pruning_ratio,test_mse,test_mae,delta_test_mse,delta_test_mae,relative_mse_change_percent,relative_mae_change_percent
0,B4_no_pruning,none,0,24,0.00,0.372546,0.398214,0.000000,0.00000,0.000000,0.000000
1,B4_static_pruning_25,static_overall_importance,6,18,0.25,0.378314,0.404424,0.005768,0.00621,1.548372,1.559421


## 15. Sonuçları kaydet

In [28]:
experiment_name = "b4_static_pruning_25"

experiment_dir = PRUNING_DIR / experiment_name
experiment_dir.mkdir(parents=True, exist_ok=True)

val_comparison.to_csv(
    experiment_dir / "validation_overall_comparison.csv",
    index=False,
)

regime_val_comparison.to_csv(
    experiment_dir / "validation_regime_comparison.csv",
    index=False,
)

static_regime_delta.to_csv(
    experiment_dir / "validation_regime_delta_static_25.csv",
    index=False,
)

test_comparison.to_csv(
    experiment_dir / "test_overall_comparison.csv",
    index=False,
)

baseline_val["window_losses"].to_csv(
    experiment_dir / "baseline_validation_window_losses.csv",
    index=False,
)

static_pruned_val["window_losses"].to_csv(
    experiment_dir / "static_pruned_validation_window_losses.csv",
    index=False,
)

static_prune_df.to_csv(
    experiment_dir / "pruned_heads.csv",
    index=False,
)

mask_df = pd.DataFrame(
    static_prune_mask.numpy(),
    index=[f"layer_{i}" for i in range(static_prune_mask.shape[0])],
    columns=[f"head_{j}" for j in range(static_prune_mask.shape[1])],
)

mask_df.to_csv(
    experiment_dir / "static_prune_mask.csv",
)

print("Saved experiment outputs to:")
print(experiment_dir)

print("\nFiles:")
for path in sorted(experiment_dir.iterdir()):
    print(path.name)

Saved experiment outputs to:
/content/drive/MyDrive/BIL401_Regime_Head_Pruning/pruning_experiments/b4_static_pruning_25

Files:
baseline_validation_window_losses.csv
pruned_heads.csv
static_prune_mask.csv
static_pruned_validation_window_losses.csv
test_overall_comparison.csv
validation_overall_comparison.csv
validation_regime_comparison.csv
validation_regime_delta_static_25.csv


## 16. Kısa yorum üret

In [29]:
static_val_row = val_comparison[
    val_comparison["setting"] == "B4_static_pruning_25"
].iloc[0]

static_test_row = test_comparison[
    test_comparison["setting"] == "B4_static_pruning_25"
].iloc[0]

print("Validation:")
print(
    f"Static 25% pruning changed validation MSE by "
    f"{static_val_row['delta_validation_mse']:.6f} "
    f"({static_val_row['relative_mse_change_percent']:.3f}%)."
)

print(
    f"Static 25% pruning changed validation MAE by "
    f"{static_val_row['delta_validation_mae']:.6f} "
    f"({static_val_row['relative_mae_change_percent']:.3f}%)."
)

print("\nTest:")
print(
    f"Static 25% pruning changed test MSE by "
    f"{static_test_row['delta_test_mse']:.6f} "
    f"({static_test_row['relative_mse_change_percent']:.3f}%)."
)

print(
    f"Static 25% pruning changed test MAE by "
    f"{static_test_row['delta_test_mae']:.6f} "
    f"({static_test_row['relative_mae_change_percent']:.3f}%)."
)

print("\nRegime-level validation delta:")
display(
    static_regime_delta[
        [
            "regime",
            "baseline_mse",
            "mse",
            "delta_mse",
            "relative_mse_change_percent",
            "baseline_mae",
            "mae",
            "delta_mae",
            "relative_mae_change_percent",
        ]
    ]
)

Validation:
Static 25% pruning changed validation MSE by -0.024409 (-3.600%).
Static 25% pruning changed validation MAE by -0.004397 (-0.792%).

Test:
Static 25% pruning changed test MSE by 0.005768 (1.548%).
Static 25% pruning changed test MAE by 0.006210 (1.559%).

Regime-level validation delta:


,regime,baseline_mse,mse,delta_mse,relative_mse_change_percent,baseline_mae,mae,delta_mae,relative_mae_change_percent
0,residual,0.685031,0.634596,-0.050435,-7.362383,0.554888,0.547463,-0.007425,-1.338145
1,seasonal,0.660362,0.689755,0.029393,4.451061,0.547936,0.561695,0.013759,2.511107
2,trend,0.679317,0.651925,-0.027392,-4.032308,0.556094,0.549721,-0.006372,-1.145920
